# Auditoría de factibilidad: straddle ATM SPY cercano a 30 días

## Objetivo

Determinar la cobertura y calidad local de pares call/put SPY con el mismo strike y vencimiento para valorar, en el futuro, el precio de un straddle ATM cercano a 30 días. Esta auditoría no construye una señal, una serie definitiva, volatilidad futura ni métricas predictivas.

## Datos y regla fijada antes del recorrido

Se leen en modo lectura las 1,674 cadenas originales compartidas de `HMM_regime_options/data/raw/` y el campo `close` **sin ajustar** de la tabla diaria local para medir cercanía ATM. Las definiciones se mantienen separadas: cobertura estricta de **27–33 DTE** y cobertura amplia de **21–45 DTE**.

Para cada sesión y definición se exige call y put con exactamente el mismo strike y vencimiento, y en ambos: bid > 0, ask ≥ bid y punto medio positivo. Se elige primero el vencimiento con DTE más cercano a 30, luego el strike más cercano al cierre SPY; los empates se resuelven por vencimiento anterior, strike menor y `contractID` léxico de call y put.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'scripts'))
from audit_atm30_straddle_feasibility import audit_atm30_straddles
print('Lógica reutilizable: scripts/audit_atm30_straddle_feasibility.py')

Lógica reutilizable: scripts/audit_atm30_straddle_feasibility.py


## Control de duplicados

Un `contractID` repetido con registros exactamente idénticos se retiene una vez; las filas adicionales solo se cuentan. Si un mismo `contractID` contiene campos diferentes, se excluyen todas sus filas de la selección. Así no se decide cuál cotización es correcta después de observar el par.

In [2]:
# Recorre las cadenas originales en modo lectura; tarda unos minutos.
audit = audit_atm30_straddles()
print(f"Cadenas recorridas: {audit['files_scanned']:,}")
display(pd.DataFrame([audit['duplicate_totals']]).rename(columns={
    'duplicate_contract_id_groups': 'IDs duplicados',
    'identical_duplicate_groups': 'grupos idénticos',
    'identical_duplicate_extra_rows': 'filas idénticas extra descartadas',
    'conflicting_contract_id_groups': 'grupos conflictivos',
    'conflicting_rows_excluded': 'filas conflictivas excluidas',
    'missing_contract_id_rows': 'filas sin contractID'
}))

Cadenas recorridas: 1,674


,IDs duplicados,grupos idénticos,filas idénticas extra descartadas,grupos conflictivos,filas conflictivas excluidas,filas sin contractID
0,9282,9282,9282,0,0,0


### Interpretación

Los conflictos se separan de las repeticiones idénticas. La cobertura de abajo se calcula después de aplicar esa regla, por lo que no depende de resolver manualmente registros en conflicto.

In [3]:
coverage = audit['coverage_by_year'].copy()
coverage['cobertura 27–33 DTE'] = 100 * coverage['strict_sessions'] / coverage['sessions']
coverage['cobertura 21–45 DTE'] = 100 * coverage['broad_sessions'] / coverage['sessions']
display(coverage.rename(columns={
    'year': 'año', 'sessions': 'sesiones de cadena',
    'strict_sessions': 'sesiones con par 27–33 DTE',
    'broad_sessions': 'sesiones con par 21–45 DTE'
})[['año', 'sesiones de cadena', 'sesiones con par 27–33 DTE', 'cobertura 27–33 DTE',
    'sesiones con par 21–45 DTE', 'cobertura 21–45 DTE']].style.format({
        'cobertura 27–33 DTE': '{:.2f}%', 'cobertura 21–45 DTE': '{:.2f}%'
    }))

for label, frame in [('27–33 DTE', audit['strict_selected']), ('21–45 DTE', audit['broad_selected'])]:
    display(Markdown(f'### Spreads seleccionados: {label}'))
    display(frame[[
        'call_spread', 'put_spread', 'call_relative_spread', 'put_relative_spread'
    ]].describe(percentiles=[.5, .95]).loc[['count', 'mean', '50%', '95%', 'max']])

,año,sesiones de cadena,sesiones con par 27–33 DTE,cobertura 27–33 DTE,sesiones con par 21–45 DTE,cobertura 21–45 DTE
0,2020,253,252,99.60%,252,99.60%
1,2021,252,252,100.00%,252,100.00%
2,2022,251,251,100.00%,251,100.00%
3,2023,250,250,100.00%,250,100.00%
4,2024,252,252,100.00%,252,100.00%
5,2025,250,250,100.00%,250,100.00%
6,2026,166,166,100.00%,166,100.00%


### Spreads seleccionados: 27–33 DTE

,call_spread,put_spread,call_relative_spread,put_relative_spread
count,1673.000000,1673.000000,1673.000000,1673.000000
mean,0.091566,0.087944,0.008935,0.009456
50%,0.050000,0.050000,0.005242,0.005900
95%,0.210000,0.230000,0.018899,0.023239
max,3.880000,3.890000,0.359943,0.509237


### Spreads seleccionados: 21–45 DTE

,call_spread,put_spread,call_relative_spread,put_relative_spread
count,1673.000000,1673.000000,1673.000000,1673.000000
mean,0.091566,0.087944,0.008935,0.009456
50%,0.050000,0.050000,0.005242,0.005900
95%,0.210000,0.230000,0.018899,0.023239
max,3.880000,3.890000,0.359943,0.509237


### Interpretación

Las dos columnas de cobertura no se mezclan: la primera responde a la ventana estrecha de aproximadamente 30 días y la segunda a una tolerancia más amplia. Los spreads se reportan para los pares elegidos por la regla fija, no como filtros nuevos.

In [4]:
# Anclas fijadas por posición cronológica: primera, sesión central y última respuesta archivada.
example_columns = [
    'position', 'anchor_date', 'status', 'source_file', 'spot_close_unadjusted',
    'expiration', 'dte', 'strike', 'call_contractID', 'put_contractID',
    'call_bid', 'call_ask', 'call_midpoint', 'call_relative_spread',
    'put_bid', 'put_ask', 'put_midpoint', 'put_relative_spread'
]
examples = audit['examples'].reindex(columns=example_columns)
display(examples)

if (examples['status'] == 'strict pair selected').all():
    display(Markdown('Las tres anclas contienen un par estricto trazable por archivo y `contractID`.'))
else:
    display(Markdown('Alguna ancla no tiene par estricto: se deja visible como ausencia, sin sustituir la fecha.'))

,position,anchor_date,status,source_file,spot_close_unadjusted,expiration,dte,strike,call_contractID,put_contractID,call_bid,call_ask,call_midpoint,call_relative_spread,put_bid,put_ask,put_midpoint,put_relative_spread
0,inicio,2020-01-02,strict pair selected,C:\Users\avazq\OneDrive\Escritorio\cienciadato...,324.87,2020-01-31,29,325.0,SPY200131C00325000,SPY200131P00325000,3.88,3.91,3.895,0.007702,3.51,3.54,3.525,0.008511
1,mitad,2023-04-28,strict pair selected,C:\Users\avazq\OneDrive\Escritorio\cienciadato...,415.93,2023-05-26,28,416.0,SPY230526C00416000,SPY230526P00416000,7.05,7.07,7.060,0.002833,5.47,5.49,5.480,0.003650
2,final,2026-08-31,strict pair selected,C:\Users\avazq\OneDrive\Escritorio\cienciadato...,767.05,2026-09-30,30,767.0,SPY260930C00767000,SPY260930P00767000,10.82,10.87,10.845,0.004610,10.03,10.08,10.055,0.004973


Las tres anclas contienen un par estricto trazable por archivo y `contractID`.

## Conclusión y límites

La celda siguiente resume cobertura y calidad únicamente para factibilidad de datos. Las cadenas no incluyen una hora de cotización/publicación verificable: toda utilización de datos de fecha t para una decisión posterior t+1 seguiría siendo un supuesto EOD. Esta auditoría no demuestra que el precio del straddle sea una señal ni que estuviera disponible a una hora concreta.

In [5]:
strict_n = len(audit['strict_selected'])
broad_n = len(audit['broad_selected'])
total_n = int(audit['coverage_by_year']['sessions'].sum())
conflicts = audit['duplicate_totals'].get('conflicting_contract_id_groups', 0)
display(Markdown(
    f"**Conclusión de factibilidad de datos locales: FACTIBLE.** "
    f"Hay {strict_n:,}/{total_n:,} ({100 * strict_n / total_n:.2f}%) sesiones con par utilizable de 27–33 DTE y "
    f"{broad_n:,}/{total_n:,} con par de 21–45 DTE; se detectaron {conflicts:,} grupos de `contractID` conflictivos, excluidos por una regla previa. "
    'La calidad temporal sigue limitada porque no hay hora de cotización verificable.'
))

**Conclusión de factibilidad de datos locales: FACTIBLE.** Hay 1,673/1,674 (99.94%) sesiones con par utilizable de 27–33 DTE y 1,673/1,674 con par de 21–45 DTE; se detectaron 0 grupos de `contractID` conflictivos, excluidos por una regla previa. La calidad temporal sigue limitada porque no hay hora de cotización verificable.

## Mis notas

- 
- 
- 